<a href="https://colab.research.google.com/github/jonasknoll57/Bachelorarbeit_Demand-AD/blob/main/V18a_Pipeline_Calibration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Kalibrierungs-Experimente (Pipeline-Validierung)

Ziel: Technisch prüfen, ob die Pipeline funktioniert. Keine systematische Evaluation.

| Check | Modelle | Zweck |
|-------|---------|-------|
| Oracle (100% Target) | AE + FC | Obere Schranke |
| Zero-Shot (MA→HD) | AE + FC | Funktioniert Transfer ohne FT? |
| Fine-Tune 5% (MA→HD) | AE + FC | Single-Source Transfer |
| Multi-Source FT 5% (MA+GI+KL→HD) | AE + FC | Multi-Source Transfer |

**Design-Entscheidungen:**
- **Scaler:** Zero-Shot nutzt Source-Scaler, alles andere Target-Scaler
- **Freezing:** AE = Encoder frozen, FC = forecast_lstm_1 frozen
- **Fine-Tune-Anteil:** 5% als Mittelwert für Kalibrierung

In [1]:
# ══════════════════════════════════════════════════════════════
# 0a — Colab Setup
# ══════════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ══════════════════════════════════════════════════════════════
# 0b — Imports
# ══════════════════════════════════════════════════════════════

import os
import math
import json
import random
import warnings
import time
import re
import glob
import pickle
import hashlib

from dataclasses import dataclass, field, asdict
from typing import List, Tuple, Dict, Optional
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from numpy.lib.stride_tricks import sliding_window_view

warnings.filterwarnings("ignore")
print(f"TF: {tf.__version__}, GPU: {tf.config.list_physical_devices('GPU')}")

TF: 2.19.0, GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# ══════════════════════════════════════════════════════════════
# 0c — V18a Config + Pfade
# ══════════════════════════════════════════════════════════════

DATA_BASE    = '/content/drive/MyDrive/BA_Colab/data'
CLEANED_BASE = '/content/drive/MyDrive/BA_Colab/cleaned'

RUN_NAME    = 'v18a_pipeline_calibration'
RESULTS_DIR = f'{DATA_BASE}/{RUN_NAME}'
MODELS_DIR  = f'{RESULTS_DIR}/models'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# V17 Modelle (Source-Modelle von Mannheim)
V17_DIR         = f'{DATA_BASE}/v17_ad_improvements/models'
V17_AE_PATH     = f'{V17_DIR}/baseline_lstm_ae.keras'
V17_FC_PATH     = f'{V17_DIR}/lstm_forecaster.keras'
V17_SCALER_PATH = f'{V17_DIR}/scaler_mannheim.pkl'

# Daten-Pfade
CITY_DEMAND_PATHS = {
    "Mannheim":       f'{CLEANED_BASE}/demand/Mannheim/demand_cleaned.parquet',
    "Heidelberg":     f'{CLEANED_BASE}/demand/Heidelberg/demand_cleaned.parquet',
    "Gießen":         f'{CLEANED_BASE}/demand/Gießen/demand_cleaned.parquet',
    "Kaiserslautern": f'{CLEANED_BASE}/demand/Kaiserslautern/demand_cleaned.parquet',
}

GEO_PATH           = f'{CLEANED_BASE}/geo_information/geo_information.parquet'
STATION_NAMES_PATH = f'{DATA_BASE}/station_names/station_names.parquet'

# Result-Cache
RESULTS_CACHE_PATH = f"{RESULTS_DIR}/results_cache.json"

# Artefakte
V18A_MODEL_FILES = {
    "CAL_AE_oracle":         f"{MODELS_DIR}/cal_ae_oracle.keras",
    "CAL_FC_oracle":         f"{MODELS_DIR}/cal_fc_oracle.keras",
    "CAL_AE_FT_5pct":        f"{MODELS_DIR}/cal_ae_ft_5pct.keras",
    "CAL_FC_FT_5pct":        f"{MODELS_DIR}/cal_fc_ft_5pct.keras",
    "CAL_AE_multi_source":   f"{MODELS_DIR}/cal_ae_multi_source.keras",
    "CAL_FC_multi_source":   f"{MODELS_DIR}/cal_fc_multi_source.keras",
    "CAL_AE_Multi_FT_5pct":  f"{MODELS_DIR}/cal_ae_multi_ft_5pct.keras",
    "CAL_FC_Multi_FT_5pct":  f"{MODELS_DIR}/cal_fc_multi_ft_5pct.keras",
}

FORCE_RETRAIN = False

In [4]:
# ══════════════════════════════════════════════════════════════
# 0d — V18 Config
# ══════════════════════════════════════════════════════════════

@dataclass
class V18Config:
    # --- Daten-Pipeline (identisch V17) ---
    aggregation_minutes: int = 60
    train_ratio: float = 0.67
    val_ratio: float = 0.82
    min_events_per_day: float = 3.0
    rolling_days: int = 56
    min_context_obs: int = 20

    # --- AE / FC Architektur (identisch V17) ---
    ae_window_size: int = 24
    ae_latent_dim: int = 32
    ae_layers: int = 2
    ae_dropout: float = 0.10
    ae_epochs: int = 50
    ae_batch_size: int = 2048
    ae_lr: float = 1e-3
    ae_early_stop: int = 8

    # --- Fine-Tuning ---
    ft_epochs: int = 30
    ft_lr: float = 1e-4
    ft_batch_size: int = 512
    ft_early_stop: int = 5

    # --- Threshold / Labeling ---
    low_demand_q: float = 0.33
    high_demand_q: float = 0.67

    # --- Pipeline ---
    require_contiguous: bool = True
    use_bidirectional: bool = False

    # --- Features (identisch V17) ---
    ae_features: List[str] = field(default_factory=lambda: [
        "n_lends",
        "n_returns",
        "hour_sin",
        "hour_cos",
        "dow_sin",
        "dow_cos",
        "month_sin",
        "month_cos",
        "is_weekend",
    ])

    score_features: List[str] = field(default_factory=lambda: [
        "n_lends",
        "n_returns",
    ])

cfg = V18Config()
ae_features = cfg.ae_features
score_features = cfg.score_features
context_steps = cfg.ae_window_size - 1

print(asdict(cfg))

{'aggregation_minutes': 60, 'train_ratio': 0.67, 'val_ratio': 0.82, 'min_events_per_day': 3.0, 'rolling_days': 56, 'min_context_obs': 20, 'ae_window_size': 24, 'ae_latent_dim': 32, 'ae_layers': 2, 'ae_dropout': 0.1, 'ae_epochs': 50, 'ae_batch_size': 2048, 'ae_lr': 0.001, 'ae_early_stop': 8, 'ft_epochs': 30, 'ft_lr': 0.0001, 'ft_batch_size': 512, 'ft_early_stop': 5, 'low_demand_q': 0.33, 'high_demand_q': 0.67, 'require_contiguous': True, 'use_bidirectional': False, 'ae_features': ['n_lends', 'n_returns', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend'], 'score_features': ['n_lends', 'n_returns']}


## Shared Functions (Daten-Pipeline + Modell-Toolbox)

Identisch zu V17 — bilden das Fundament des AD-Modells.

In [5]:
# ══════════════════════════════════════════════════════════════
# 1 — Prepared City Data Cache
# ══════════════════════════════════════════════════════════════

PREPARED_CACHE_ROOT = Path(f"{CLEANED_BASE}/prepared_city_data")
PREPARED_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

def _safe_city_name(city_name: str) -> str:
    return str(city_name).strip().replace("/", "_").replace("\\", "_")

def _build_prepare_signature(cfg):
    relevant = {
        "aggregation_minutes": cfg.aggregation_minutes,
        "min_events_per_day": cfg.min_events_per_day,
        "low_demand_q": cfg.low_demand_q,
        "high_demand_q": cfg.high_demand_q,
        "train_ratio": cfg.train_ratio,
        "val_ratio": cfg.val_ratio,
    }
    raw = json.dumps(relevant, sort_keys=True, default=str)
    return hashlib.md5(raw.encode()).hexdigest()[:10]

def build_prepared_cache_paths(city_name, cfg):
    city_dir = PREPARED_CACHE_ROOT / _safe_city_name(city_name)
    city_dir.mkdir(parents=True, exist_ok=True)
    sig = _build_prepare_signature(cfg)
    return (
        city_dir / f"prepared_{sig}.parquet",
        city_dir / f"station_profile_{sig}.parquet",
        city_dir / f"meta_{sig}.json",
    )

def load_prepared_city_cache(city_name, cfg):
    prepared_path, profile_path, meta_path = build_prepared_cache_paths(city_name, cfg)
    if prepared_path.exists() and profile_path.exists() and meta_path.exists():
        try:
            prepared = pd.read_parquet(prepared_path)
            profile = pd.read_parquet(profile_path)
            with open(meta_path, "r") as f:
                meta = json.load(f)
            train_end = pd.to_datetime(meta["train_end"], utc=True)
            val_end = pd.to_datetime(meta["val_end"], utc=True)
            print(f"  ✓ Prepared-Cache geladen: {city_name} ({prepared_path.name})")
            return prepared, profile, train_end, val_end
        except Exception as e:
            print(f"  WARNUNG: Cache für {city_name} unlesbar ({e}) — baue neu.")
    return None

def save_prepared_city_cache(city_name, cfg, prepared, profile, train_end, val_end):
    prepared_path, profile_path, meta_path = build_prepared_cache_paths(city_name, cfg)
    prepared.to_parquet(prepared_path, index=False)
    profile.to_parquet(profile_path, index=False)
    with open(meta_path, "w") as f:
        json.dump(
            {
                "city_name": city_name,
                "signature": _build_prepare_signature(cfg),
                "train_end": str(train_end),
                "val_end": str(val_end),
                "n_rows_prepared": int(len(prepared)),
                "n_stations_profile": int(profile["station_id"].nunique()) if len(profile) else 0,
            },
            f,
            indent=2,
            default=str,
        )
    print(f"  ✓ Prepared-Cache gespeichert: {city_name}")

In [6]:
# ══════════════════════════════════════════════════════════════
# 2 — Aggregation
# ══════════════════════════════════════════════════════════════

def aggregate_station_level(df: pd.DataFrame, minutes: int = 60) -> pd.DataFrame:
    df = df.copy()

    required_cols = {"timestamp", "station_id", "station_name_id", "n_lends", "n_returns"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"aggregate_station_level: Fehlende Spalten: {sorted(missing)}")

    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
    df = df.dropna(subset=["timestamp", "station_id"]).copy()

    df["n_lends"] = pd.to_numeric(df["n_lends"], errors="coerce").fillna(0).astype(np.int32)
    df["n_returns"] = pd.to_numeric(df["n_returns"], errors="coerce").fillna(0).astype(np.int32)
    df["hour_ts"] = df["timestamp"].dt.floor(f"{minutes}min")

    agg_dict = {
        "n_lends": ("n_lends", "sum"),
        "n_returns": ("n_returns", "sum"),
        "station_name_id": ("station_name_id", "first"),
    }

    for col in ["station_name", "latitude", "longitude", "location_id", "network_name"]:
        if col in df.columns:
            agg_dict[col] = (col, "first")

    agg = (
        df.groupby(["station_id", "hour_ts"], as_index=False)
          .agg(**agg_dict)
          .sort_values(["station_id", "hour_ts"])
          .reset_index(drop=True)
    )

    agg["total_demand"] = agg["n_lends"] + agg["n_returns"]
    agg["net_flow"] = agg["n_returns"] - agg["n_lends"]
    agg["abs_net_flow"] = agg["net_flow"].abs()

    return agg

In [7]:
# ══════════════════════════════════════════════════════════════
# 3 — Gap-Fill
# ══════════════════════════════════════════════════════════════

def fill_missing_time_bins(x: pd.DataFrame, minutes: int = 60) -> pd.DataFrame:
    x = x.copy()
    x["hour_ts"] = pd.to_datetime(x["hour_ts"], utc=True, errors="coerce")
    x = x.dropna(subset=["station_id", "hour_ts"]).copy()

    # Defensive Duplikatbehandlung
    dup_mask = x.duplicated(subset=["station_id", "hour_ts"], keep=False)
    if dup_mask.any():
        print(f"  WARNUNG: {int(dup_mask.sum()):,} Duplikate vor Gap-Fill")
        agg_dict = {}
        for col in ["n_lends", "n_returns", "total_demand", "net_flow", "abs_net_flow"]:
            if col in x.columns:
                agg_dict[col] = (col, "sum")
        for col in ["station_name_id", "station_name", "latitude", "longitude", "location_id", "network_name"]:
            if col in x.columns:
                agg_dict[col] = (col, "first")

        x = (
            x.groupby(["station_id", "hour_ts"], as_index=False)
             .agg(**agg_dict)
             .sort_values(["station_id", "hour_ts"])
             .reset_index(drop=True)
        )

    full_range = pd.date_range(
        start=x["hour_ts"].min(),
        end=x["hour_ts"].max(),
        freq=f"{minutes}min",
        tz="UTC",
    )

    parts = []
    meta_first_cols = [
        c for c in [
            "station_name_id",
            "station_name",
            "latitude",
            "longitude",
            "location_id",
            "network_name",
        ] if c in x.columns
    ]

    for sid, g in x.groupby("station_id", sort=False):
        g = g.sort_values("hour_ts").copy()
        g = g.set_index("hour_ts").reindex(full_range)
        g.index.name = "hour_ts"
        g["station_id"] = sid

        for c in ["n_lends", "n_returns", "total_demand", "net_flow", "abs_net_flow"]:
            if c in g.columns:
                g[c] = g[c].fillna(0)

        for c in meta_first_cols:
            g[c] = g[c].ffill().bfill()

        parts.append(g.reset_index())

    out = pd.concat(parts, ignore_index=True).sort_values(["station_id", "hour_ts"]).reset_index(drop=True)

    if "total_demand" not in out.columns:
        out["total_demand"] = out["n_lends"] + out["n_returns"]
    if "net_flow" not in out.columns:
        out["net_flow"] = out["n_returns"] - out["n_lends"]
    if "abs_net_flow" not in out.columns:
        out["abs_net_flow"] = out["net_flow"].abs()

    return out

In [8]:
# ══════════════════════════════════════════════════════════════
# 4 — Station-Filter, Demand-Regime, Train/Val/Test Split
# ══════════════════════════════════════════════════════════════

def prepare_stations_and_splits(df: pd.DataFrame, cfg, city_name=""):
    n_days = (df["hour_ts"].max() - df["hour_ts"].min()).days + 1
    min_events = int(n_days * cfg.min_events_per_day)

    station_volume = df.groupby("station_id")["total_demand"].sum()
    active_ids = station_volume[station_volume >= min_events].index.tolist()
    df = df[df["station_id"].isin(active_ids)].copy()

    station_profile = (
        df.groupby(["station_id", "station_name"], as_index=False)
          .agg(
              avg_total_demand_h=("total_demand", "mean"),
              avg_lends_h=("n_lends", "mean"),
              avg_returns_h=("n_returns", "mean"),
              latitude=("latitude", "first"),
              longitude=("longitude", "first"),
          )
    )

    q1 = station_profile["avg_total_demand_h"].quantile(cfg.low_demand_q)
    q2 = station_profile["avg_total_demand_h"].quantile(cfg.high_demand_q)

    station_profile["demand_regime"] = station_profile["avg_total_demand_h"].apply(
        lambda x: "low" if x <= q1 else ("mid" if x <= q2 else "high")
    )

    df = df.merge(
        station_profile[["station_id", "demand_regime"]],
        on="station_id",
        how="left",
    )

    t_min = df["hour_ts"].min()
    t_max = df["hour_ts"].max()
    total_seconds = (t_max - t_min).total_seconds()

    train_end = t_min + pd.Timedelta(seconds=total_seconds * cfg.train_ratio)
    val_end   = t_min + pd.Timedelta(seconds=total_seconds * cfg.val_ratio)

    print(
        f"  {city_name}: {df['station_id'].nunique()} aktive Stationen | "
        f"Train bis {train_end} | Val bis {val_end}"
    )

    return df, station_profile, train_end, val_end

In [9]:
# ══════════════════════════════════════════════════════════════
# 5 — Statistik-Pipeline (Kontext-Scores, Poisson, ECDF, Labels)
# ══════════════════════════════════════════════════════════════

TARGETS = ["n_lends", "n_returns", "total_demand", "net_flow"]
COUNT_TARGETS = ["n_lends", "n_returns", "total_demand"]

def add_context_keys(df):
    df = df.copy()
    df["hour"] = df["hour_ts"].dt.hour
    df["dow"] = df["hour_ts"].dt.dayofweek
    df["is_wknd"] = (df["dow"] >= 5).astype(int)
    return df

def rolling_context_scores_vectorized(df, target, rolling_days=56, min_obs=20):
    df = df.copy().sort_values(["station_id", "hour_ts"])
    window_h = rolling_days * 24

    g = df.groupby(["station_id", "hour", "is_wknd"])[target]
    roll_mean = g.transform(lambda s: s.rolling(window_h, min_periods=min_obs).mean())
    roll_std  = g.transform(lambda s: s.rolling(window_h, min_periods=min_obs).std()).clip(lower=1e-6)

    df[f"{target}_z_ctx_roll"] = (df[target] - roll_mean) / roll_std
    return df

def add_rolling_poisson_scores_vectorized(df, target, rolling_days=56, min_obs=20):
    from scipy.stats import poisson

    df = df.copy().sort_values(["station_id", "hour_ts"])
    window_h = rolling_days * 24

    g = df.groupby(["station_id", "hour", "is_wknd"])[target]
    lam = g.transform(lambda s: s.rolling(window_h, min_periods=min_obs).mean()).clip(lower=1e-6)

    vals = df[target].clip(lower=0).astype(int)
    pvals = poisson.sf(vals - 1, lam)

    df[f"{target}_poisson_sf_roll"] = -np.log10(np.clip(pvals, 1e-12, 1.0))
    return df

def add_ecdf_surprisal_scores(df, target):
    df = df.copy().sort_values(["station_id", "hour_ts"])

    def _ecdf_surprisal(s):
        ranks = s.rank(method="average", pct=True)
        tail = np.minimum(ranks, 1 - ranks + 1e-12)
        return -np.log10(np.clip(tail * 2, 1e-12, 1.0))

    df[f"{target}_ecdf_surprisal"] = (
        df.groupby(["station_id", "hour", "is_wknd"])[target]
          .transform(_ecdf_surprisal)
    )
    return df

def add_statistical_labels(df):
    df = df.copy()

    label_cols = []
    for target in TARGETS:
        z_col = f"{target}_z_ctx_roll"
        if z_col in df.columns:
            df[f"flag_{target}_z"] = (df[z_col].abs() >= 3).astype(int)
            label_cols.append(f"flag_{target}_z")

    for target in COUNT_TARGETS:
        p_col = f"{target}_poisson_sf_roll"
        if p_col in df.columns:
            df[f"flag_{target}_poisson"] = (df[p_col] >= 3).astype(int)
            label_cols.append(f"flag_{target}_poisson")

    for target in TARGETS:
        e_col = f"{target}_ecdf_surprisal"
        if e_col in df.columns:
            df[f"flag_{target}_ecdf"] = (df[e_col] >= 2).astype(int)
            label_cols.append(f"flag_{target}_ecdf")

    if label_cols:
        df["stat_anomaly_label"] = (df[label_cols].sum(axis=1) > 0).astype(int)
    else:
        df["stat_anomaly_label"] = 0

    return df

In [10]:
# ══════════════════════════════════════════════════════════════
# 6 — Sequenzbuilder (Window-Level Labels)
# ══════════════════════════════════════════════════════════════

def make_sequences_with_window_labels(
    x: pd.DataFrame,
    feature_cols: List[str],
    window_size: int,
    synth_label_col: str = "synth_label",
    synth_type_col: str = "synth_type",
    require_contiguous: bool = True,
    agg_minutes: int = 60,
) -> Tuple[np.ndarray, pd.DataFrame]:
    X_parts, meta_parts = [], []
    expected_ns = pd.Timedelta(minutes=agg_minutes).value

    for sid, g in x.groupby("station_id", sort=False):
        g = g.sort_values("hour_ts").reset_index(drop=True)
        n = len(g)
        if n < window_size:
            continue

        vals = g[feature_cols].to_numpy(dtype=np.float32)
        win = sliding_window_view(vals, window_shape=window_size, axis=0)
        win = np.moveaxis(win, -1, 1)

        valid_mask = np.ones(n - window_size + 1, dtype=bool)

        if require_contiguous:
            ts_int = pd.to_datetime(g["hour_ts"]).astype("int64").to_numpy()
            diffs = np.diff(ts_int)
            step_ok = (diffs == expected_ns).astype(np.int8)
            if window_size > 1:
                ok_sums = np.convolve(step_ok, np.ones(window_size - 1, dtype=np.int32), mode="valid")
                valid_mask = (ok_sums == (window_size - 1))

        if not valid_mask.any():
            continue

        X_valid = win[valid_mask]

        end_idx = np.arange(window_size - 1, n)[valid_mask]
        last_rows = g.iloc[end_idx].copy()

        if synth_label_col in g.columns:
            label_last = g[synth_label_col].iloc[end_idx].fillna(0).astype(int).to_numpy()
        else:
            label_last = np.zeros(len(end_idx), dtype=int)

        if synth_type_col in g.columns:
            type_last = g[synth_type_col].iloc[end_idx].fillna("normal").astype(str).to_numpy()
        else:
            type_last = np.array(["normal"] * len(end_idx), dtype=object)

        meta = last_rows.copy()
        meta["label_last"] = label_last
        meta["type_last"] = type_last

        X_parts.append(X_valid)
        meta_parts.append(meta)

    if not X_parts:
        return np.empty((0, window_size, len(feature_cols)), dtype=np.float32), pd.DataFrame()

    X = np.concatenate(X_parts, axis=0)
    meta = pd.concat(meta_parts, axis=0).reset_index(drop=True)

    meta["label_eval"] = np.where(meta["label_last"] == 1, "anomaly", "normal")
    meta["type_eval"] = meta["type_last"]

    return X, meta

In [11]:
# ══════════════════════════════════════════════════════════════
# 7 — Modell-Architekturen (identisch V17)
# ══════════════════════════════════════════════════════════════

def build_lstm_autoencoder(
    n_input_features,
    window_size,
    latent_dim=32,
    n_layers=2,
    dropout=0.1,
    bidirectional=False,
    n_output_features=None,
):
    n_output_features = n_output_features or n_input_features

    inputs = keras.Input(shape=(window_size, n_input_features), name="encoder_input")
    x = inputs

    for i in range(n_layers):
        return_sequences = (i < n_layers - 1)
        lstm = layers.LSTM(
            latent_dim,
            return_sequences=return_sequences,
            dropout=dropout,
            name=f"encoder_lstm_{i+1}",
        )
        x = layers.Bidirectional(lstm, name=f"encoder_bi_{i+1}")(x) if bidirectional else lstm(x)

    x = layers.RepeatVector(window_size, name="repeat_vector")(x)

    for i in range(n_layers):
        lstm = layers.LSTM(
            latent_dim,
            return_sequences=True,
            dropout=dropout,
            name=f"decoder_lstm_{i+1}",
        )
        x = layers.Bidirectional(lstm, name=f"decoder_bi_{i+1}")(x) if bidirectional else lstm(x)

    outputs = layers.TimeDistributed(
        layers.Dense(n_output_features),
        name="reconstruction_output",
    )(x)

    return keras.Model(inputs, outputs, name="lstm_autoencoder")

def build_lstm_forecaster(
    n_input_features,
    n_target_features,
    context_steps,
    latent_dim=32,
    n_layers=2,
    dropout=0.1,
):
    inputs = keras.Input(shape=(context_steps, n_input_features), name="forecast_input")
    x = inputs

    for i in range(n_layers):
        return_sequences = (i < n_layers - 1)
        x = layers.LSTM(
            latent_dim,
            return_sequences=return_sequences,
            dropout=dropout,
            name=f"forecast_lstm_{i+1}",
        )(x)

    outputs = layers.Dense(n_target_features, name="forecast_output")(x)
    return keras.Model(inputs, outputs, name="lstm_forecaster")

In [12]:
# ══════════════════════════════════════════════════════════════
# 8 — Training, Scoring, Evaluation Helpers
# ══════════════════════════════════════════════════════════════

def train_model_generic(
    model,
    X_train,
    Y_train,
    X_val,
    Y_val,
    epochs,
    lr,
    batch_size,
    early_stop,
    verbose=1,
):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0),
        loss="mse",
    )

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=early_stop,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=4,
            verbose=1,
        ),
    ]

    history = model.fit(
        X_train,
        Y_train,
        validation_data=(X_val, Y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=verbose,
    )
    return model, history

def load_or_train(model_path, build_fn, force=False):
    if os.path.exists(model_path) and not force:
        print(f"  ✓ Lade aus Cache: {os.path.basename(model_path)}")
        return keras.models.load_model(model_path), None

    print(f"  ▶ Trainiere neu: {os.path.basename(model_path)}")
    model, history = build_fn()
    model.save(model_path)
    print(f"  ✓ Gespeichert: {os.path.basename(model_path)}")
    return model, history

def predict_in_batches(model, X, batch_size=4096):
    return model.predict(X, batch_size=batch_size, verbose=0)

def compute_reconstruction_scores(X_true, X_pred, feature_cols, score_features):
    idx = [feature_cols.index(f) for f in score_features]
    err = (X_true[:, :, idx] - X_pred[:, :, idx]) ** 2
    return err.mean(axis=(1, 2))

def compute_forecast_scores(model, X_input, X_last_full, score_features, ae_features):
    idx = [ae_features.index(f) for f in score_features]
    y_pred = model.predict(X_input, batch_size=4096, verbose=0)
    y_true = X_last_full[:, idx]
    return ((y_true - y_pred) ** 2).mean(axis=1)

def znorm_score_by_hour(meta, score_col, train_end, val_end, out_col):
    meta = meta.copy()
    train_mask = meta["hour_ts"] < train_end

    stats = (
        meta.loc[train_mask]
            .groupby("hour")[score_col]
            .agg(["mean", "std"])
            .reset_index()
            .rename(columns={"mean": "_mu", "std": "_sd"})
    )

    stats["_sd"] = stats["_sd"].fillna(0).clip(lower=1e-6)
    meta = meta.merge(stats, on="hour", how="left")
    meta[out_col] = (meta[score_col] - meta["_mu"]) / meta["_sd"]

    return meta.drop(columns=["_mu", "_sd"]), stats, None

In [13]:
# ══════════════════════════════════════════════════════════════
# 9 — Evaluation (vereinfacht fuer Kalibrierung)
# ══════════════════════════════════════════════════════════════

EVAL_ANOMALY_TYPES = ["point_spike", "point_drop", "collective"]
EVAL_REGIMES = ["high", "mid", "low"]

def evaluate_v18(meta, score_col, experiment_name, val_start, test_start, verbose=True):
    meta = meta.copy()

    meta["split_eval"] = np.where(
        meta["hour_ts"] < val_start,
        "train",
        np.where(meta["hour_ts"] < test_start, "val", "test"),
    )

    val_m = meta[meta["split_eval"] == "val"].copy()
    test_m = meta[meta["split_eval"] == "test"].copy()

    if len(val_m) == 0 or len(test_m) == 0:
        print(f"  [{experiment_name}] WARNUNG: Keine VAL/TEST-Daten!")
        return {}

    results = {
        "experiment": experiment_name,
        "n_val": len(val_m),
        "n_test": len(test_m),
    }

    # Threshold auf VAL
    y_val = val_m["synth_label"].astype(int).values
    s_val = val_m[score_col].values

    threshold = None
    if len(np.unique(y_val)) > 1:
        prec_v, rec_v, thr_v = precision_recall_curve(y_val, s_val)
        results["val_pr_auc"] = average_precision_score(y_val, s_val)

        if len(thr_v) > 0:
            f1_v = 2 * prec_v[:-1] * rec_v[:-1] / np.clip(prec_v[:-1] + rec_v[:-1], 1e-12, None)
            best_idx = int(np.nanargmax(f1_v))
            threshold = float(thr_v[best_idx])
            results["val_best_f1"] = float(f1_v[best_idx])

    if threshold is None:
        threshold = float(np.quantile(s_val, 0.99)) if len(s_val) else 0.0

    results["threshold"] = threshold

    # Testmetriken
    y_test = test_m["synth_label"].astype(int).values
    s_test = test_m[score_col].values
    y_hat = (s_test >= threshold).astype(int)

    if len(np.unique(y_test)) > 1:
        results["pr_auc"] = float(average_precision_score(y_test, s_test))
        try:
            results["roc_auc"] = float(roc_auc_score(y_test, s_test))
        except Exception:
            results["roc_auc"] = None
    else:
        results["pr_auc"] = None
        results["roc_auc"] = None

    results["f1"] = float(f1_score(y_test, y_hat, zero_division=0))
    results["precision"] = float(precision_score(y_test, y_hat, zero_division=0))
    results["recall"] = float(recall_score(y_test, y_hat, zero_division=0))

    # Per-Type
    per_type = {}
    for a_type in EVAL_ANOMALY_TYPES:
        sub = test_m[test_m["synth_type"] == a_type]
        if len(sub) == 0:
            continue
        yt = sub["synth_label"].astype(int).values
        yh = (sub[score_col].values >= threshold).astype(int)
        per_type[a_type] = {
            "n": int(len(sub)),
            "recall": float(recall_score(yt, yh, zero_division=0)),
        }
    results["per_type"] = per_type

    # Per-Regime
    per_regime = {}
    for reg in EVAL_REGIMES:
        sub = test_m[test_m["demand_regime"] == reg]
        if len(sub) == 0:
            continue
        yt = sub["synth_label"].astype(int).values
        yh = (sub[score_col].values >= threshold).astype(int)
        per_regime[reg] = {
            "n": int(len(sub)),
            "recall": float(recall_score(yt, yh, zero_division=0)),
        }
    results["per_regime"] = per_regime

    if verbose:
        print(f"\n[{experiment_name}]")
        print(f"  VAL:  n={len(val_m):,}, positives={int(y_val.sum()):,}")
        print(f"  TEST: n={len(test_m):,}, positives={int(y_test.sum()):,}")
        print(f"  Threshold: {threshold:.6f}")
        print(f"  PR-AUC:    {results['pr_auc']}")
        print(f"  F1:        {results['f1']:.4f}")
        print(f"  Precision: {results['precision']:.4f}")
        print(f"  Recall:    {results['recall']:.4f}")

    return results

In [14]:
# ══════════════════════════════════════════════════════════════
# 10 — Transfer Functions + Injection + Data Helpers
# ══════════════════════════════════════════════════════════════

def freeze_ae_encoder(model):
    for layer in model.layers:
        if "encoder" in layer.name:
            layer.trainable = False

    trainable = sum(p.numpy().size for p in model.trainable_weights)
    total = sum(p.numpy().size for p in model.weights)
    print(f"  AE Freeze: {total-trainable:,} frozen, {trainable:,} trainable")
    return model

def freeze_fc_first_lstm(model):
    for layer in model.layers:
        if layer.name == "forecast_lstm_1":
            layer.trainable = False

    trainable = sum(p.numpy().size for p in model.trainable_weights)
    total = sum(p.numpy().size for p in model.weights)
    print(f"  FC Freeze: {total-trainable:,} frozen, {trainable:,} trainable")
    return model

def fine_tune_model(
    model,
    X_train,
    Y_train,
    X_val,
    Y_val,
    epochs=30,
    lr=1e-4,
    batch_size=512,
    early_stop=5,
):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0),
        loss="mse",
    )
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=early_stop,
            restore_best_weights=True,
            verbose=1,
        )
    ]

    history = model.fit(
        X_train,
        Y_train,
        validation_data=(X_val, Y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=1,
    )
    return model, history

freeze_fc_backbone = freeze_fc_first_lstm

def inject_synthetic_anomalies_targeted_v2(
    df,
    split_col="split",
    label_col="synth_label",
    type_col="synth_type",
    random_state=42,
):
    rng = np.random.default_rng(random_state)
    out = df.copy()

    out[label_col] = 0
    out[type_col] = "normal"

    test_mask = out[split_col] == "test"
    test_idx = out.index[test_mask].to_numpy()

    if len(test_idx) == 0:
        return out

    n_total = len(test_idx)
    n_point_spike = max(1, int(n_total * 0.004))
    n_point_drop  = max(1, int(n_total * 0.001))
    n_collective  = max(1, int(n_total * 0.004))

    # Point spikes
    spike_idx = rng.choice(test_idx, size=min(n_point_spike, len(test_idx)), replace=False)
    out.loc[spike_idx, "n_lends"] = out.loc[spike_idx, "n_lends"] * 4 + 3
    out.loc[spike_idx, "n_returns"] = out.loc[spike_idx, "n_returns"] * 4 + 3
    out.loc[spike_idx, label_col] = 1
    out.loc[spike_idx, type_col] = "point_spike"

    remaining = np.setdiff1d(test_idx, spike_idx)

    # Point drops
    cand_drop = out.loc[remaining]
    cand_drop = cand_drop[cand_drop["total_demand"] >= 5]
    drop_idx = rng.choice(cand_drop.index.to_numpy(), size=min(n_point_drop, len(cand_drop)), replace=False) if len(cand_drop) else np.array([], dtype=int)

    if len(drop_idx):
        out.loc[drop_idx, "n_lends"] = 0
        out.loc[drop_idx, "n_returns"] = 0
        out.loc[drop_idx, label_col] = 1
        out.loc[drop_idx, type_col] = "point_drop"

    remaining = np.setdiff1d(remaining, drop_idx)

    # Collective
    n_blocks = max(1, n_collective // 6)
    chosen_collective = []

    for sid, g in out.loc[remaining].groupby("station_id"):
        g = g.sort_values("hour_ts")
        idxs = g.index.to_numpy()
        if len(idxs) < 6:
            continue

        possible_starts = np.arange(0, len(idxs) - 5)
        rng.shuffle(possible_starts)

        used = 0
        for s in possible_starts:
            block = idxs[s:s+6]
            if np.all(out.loc[block, label_col].values == 0):
                out.loc[block, "n_lends"] = out.loc[block, "n_lends"] * 3 + 2
                out.loc[block, "n_returns"] = out.loc[block, "n_returns"] * 3 + 2
                out.loc[block, label_col] = 1
                out.loc[block, type_col] = "collective"
                chosen_collective.extend(block.tolist())
                used += 1
            if used >= n_blocks:
                break
        if used >= n_blocks:
            break

    out["total_demand"] = out["n_lends"] + out["n_returns"]
    out["net_flow"] = out["n_returns"] - out["n_lends"]
    out["abs_net_flow"] = out["net_flow"].abs()

    return out

def fit_city_scaler(df, train_end, feature_cols, city_name):
    train_df = df[df["hour_ts"] < train_end].copy()
    scaler = StandardScaler()
    scaler.fit(train_df[feature_cols].values)
    print(f"  ✓ Scaler fit: {city_name} | n={len(train_df):,}")
    return scaler

def apply_feature_engineering(df):
    df = df.copy()
    ts = pd.to_datetime(df["hour_ts"], utc=True)

    hour = ts.dt.hour
    dow = ts.dt.dayofweek
    month = ts.dt.month

    df["hour"] = hour
    df["dow"] = dow
    df["month"] = month
    df["is_weekend"] = (dow >= 5).astype(int)

    df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    df["dow_sin"] = np.sin(2 * np.pi * dow / 7)
    df["dow_cos"] = np.cos(2 * np.pi * dow / 7)
    df["month_sin"] = np.sin(2 * np.pi * month / 12)
    df["month_cos"] = np.cos(2 * np.pi * month / 12)

    return df

def add_split_column(df, train_end, val_end):
    df = df.copy()
    df["split"] = np.where(
        df["hour_ts"] < train_end,
        "train",
        np.where(df["hour_ts"] < val_end, "val", "test"),
    )
    return df

def get_last_fraction(X, Y, frac):
    n = len(X)
    k = max(1, int(n * frac))
    return X[-k:], Y[-k:]

def get_last_fraction_fc(X, Y, frac):
    n = len(X)
    k = max(1, int(n * frac))
    return X[-k:], Y[-k:]

In [15]:
# ══════════════════════════════════════════════════════════════
# 11 — Results Cache + City Data Loader + Sequence Builder
# ══════════════════════════════════════════════════════════════

def save_result(results_dict, key, result, cache_path=RESULTS_CACHE_PATH):
    result_clean = {
        k: v
        for k, v in result.items()
        if k not in ("meta", "cross_matrix") and not isinstance(v, pd.DataFrame)
    }
    results_dict[key] = result_clean

    with open(cache_path, "w") as f:
        json.dump(results_dict, f, indent=2, default=str)

    return results_dict

def load_results_cache(cache_path=RESULTS_CACHE_PATH):
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            return json.load(f)
    return {}

def prepare_city_data(demand_path, geo_path, station_names_path, cfg, city_name, force_rebuild=False):
    print(f"\n{'='*70}")
    print(f"  DATEN-PIPELINE: {city_name}")
    print(f"{'='*70}")

    if not force_rebuild:
        cached = load_prepared_city_cache(city_name, cfg)
        if cached is not None:
            return cached

    demand = pd.read_parquet(demand_path)
    geo = pd.read_parquet(geo_path)
    snames = pd.read_parquet(station_names_path)

    geo = geo[["location_id", "latitude", "longitude"]].drop_duplicates()
    snames = snames.rename(columns={"id": "station_name_id", "name": "station_name"})

    df = demand.merge(geo, on="location_id", how="left")
    df = df.merge(snames[["station_name_id", "station_name"]], on="station_name_id", how="left")

    df = aggregate_station_level(df, minutes=cfg.aggregation_minutes)
    df = fill_missing_time_bins(df, minutes=cfg.aggregation_minutes)
    df = prepare_stations_and_splits(df, cfg, city_name=city_name)

    prepared, station_profile, train_end, val_end = df
    prepared = add_context_keys(prepared)

    for target in TARGETS:
        prepared = rolling_context_scores_vectorized(
            prepared,
            target,
            rolling_days=cfg.rolling_days,
            min_obs=cfg.min_context_obs,
        )

    for target in COUNT_TARGETS:
        prepared = add_rolling_poisson_scores_vectorized(
            prepared,
            target,
            rolling_days=cfg.rolling_days,
            min_obs=cfg.min_context_obs,
        )

    for target in TARGETS:
        prepared = add_ecdf_surprisal_scores(prepared, target)

    prepared = add_statistical_labels(prepared)
    prepared = apply_feature_engineering(prepared)

    save_prepared_city_cache(city_name, cfg, prepared, station_profile, train_end, val_end)
    return prepared, station_profile, train_end, val_end

def build_city_sequences(df, scaler, feature_cols, cfg, city_short="CITY"):
    x = df.copy()
    x[feature_cols] = scaler.transform(x[feature_cols].values)

    X_all, meta_all = make_sequences_with_window_labels(
        x,
        feature_cols=feature_cols,
        window_size=cfg.ae_window_size,
        synth_label_col="synth_label",
        synth_type_col="synth_type",
        require_contiguous=cfg.require_contiguous,
        agg_minutes=cfg.aggregation_minutes,
    )

    print(f"  ✓ {city_short}: Sequenzen gebaut: {X_all.shape}")
    return X_all, meta_all

def load_or_create_injected_dataset(df_hd, train_end_hd, cfg, city_name="Heidelberg"):
    inj_dir = Path(f"{CLEANED_BASE}/prepared_city_data/{city_name}")
    inj_dir.mkdir(parents=True, exist_ok=True)

    sig = _build_prepare_signature(cfg)
    inj_path = inj_dir / f"injected_{sig}.parquet"

    if inj_path.exists():
        print(f"  ✓ Injiziertes Dataset geladen: {inj_path.name}")
        return pd.read_parquet(inj_path)

    print("  ▶ Erzeuge neues injiziertes Target-Dataset...")
    x = add_split_column(df_hd, train_end_hd, val_end_hd)
    x = inject_synthetic_anomalies_targeted_v2(x)

    x.to_parquet(inj_path, index=False)
    print(f"  ✓ Injiziertes Dataset gespeichert: {inj_path.name}")
    return x

results_all = load_results_cache()

## Schritt 1: Staedte laden + Sequenzen bauen

Alle 4 Staedte werden geladen (mit Cache). Nur HD bekommt Injection.

In [20]:
# ══════════════════════════════════════════════════════════════
# 12 — Alle Staedte laden
# ══════════════════════════════════════════════════════════════

city_data = {}

for city_name in ["Mannheim", "Heidelberg", "Gießen", "Kaiserslautern"]:
    city_data[city_name] = prepare_city_data(
        CITY_DEMAND_PATHS[city_name],
        GEO_PATH,
        STATION_NAMES_PATH,
        cfg,
        city_name,
    )

df_ma, profile_ma, train_end_ma, val_end_ma = city_data["Mannheim"]
df_hd, profile_hd, train_end_hd, val_end_hd = city_data["Heidelberg"]
df_gi, profile_gi, train_end_gi, val_end_gi = city_data["Gießen"]
df_kl, profile_kl, train_end_kl, val_end_kl = city_data["Kaiserslautern"]

# Staedtevergleich
print("\n" + "=" * 70 + "\n  STAEDTEVERGLEICH\n" + "=" * 70)
for name, (df_c, prof, _, _) in city_data.items():
    n_st = df_c["station_id"].nunique()
    n_days = (df_c["hour_ts"].max() - df_c["hour_ts"].min()).days
    avg_d = df_c["total_demand"].mean()
    print(f"  {name:18s}: {n_st:3d} Stationen, {n_days:4d} Tage, avg_demand/h={avg_d:.2f}")


  DATEN-PIPELINE: Mannheim
  ✓ Prepared-Cache geladen: Mannheim (prepared_fd164b228a.parquet)

  DATEN-PIPELINE: Heidelberg
  ✓ Prepared-Cache geladen: Heidelberg (prepared_fd164b228a.parquet)

  DATEN-PIPELINE: Gießen
  ✓ Prepared-Cache geladen: Gießen (prepared_fd164b228a.parquet)

  DATEN-PIPELINE: Kaiserslautern
  ✓ Prepared-Cache geladen: Kaiserslautern (prepared_fd164b228a.parquet)

  STAEDTEVERGLEICH
  Mannheim          :  88 Stationen, 1039 Tage, avg_demand/h=1.68
  Heidelberg        :  58 Stationen, 1039 Tage, avg_demand/h=2.01
  Gießen            :  43 Stationen, 1039 Tage, avg_demand/h=1.50
  Kaiserslautern    :  23 Stationen, 1039 Tage, avg_demand/h=0.75


In [20]:
# ══════════════════════════════════════════════════════════════
# 13 — Injection auf Target (Heidelberg) + Scaler + Sequenzen
# ══════════════════════════════════════════════════════════════

# Injection
df_hd_inj = load_or_create_injected_dataset(df_hd, train_end_hd, cfg, "Heidelberg")

# Scaler pro Stadt
scaler_ma = fit_city_scaler(df_ma, train_end_ma, ae_features, "Mannheim")
scaler_hd = fit_city_scaler(df_hd, train_end_hd, ae_features, "Heidelberg")
scaler_gi = fit_city_scaler(df_gi, train_end_gi, ae_features, "Gießen")
scaler_kl = fit_city_scaler(df_kl, train_end_kl, ae_features, "Kaiserslautern")

score_idx = [ae_features.index(f) for f in score_features]

# --- Mannheim (Source) ---
print("\n--- Mannheim (Source) ---")
X_ma_all, meta_ma_all = build_city_sequences(df_ma, scaler_ma, ae_features, cfg, "MA")

meta_ma_all["split_eval"] = np.where(
    meta_ma_all["hour_ts"] < train_end_ma,
    "train",
    np.where(meta_ma_all["hour_ts"] < val_end_ma, "val", "test"),
)

train_normal_ma = (meta_ma_all["split_eval"] == "train") & (meta_ma_all["label_eval"] == "normal")
val_normal_ma   = (meta_ma_all["split_eval"] == "val") & (meta_ma_all["label_eval"] == "normal")

X_ma_train = X_ma_all[train_normal_ma.values]
X_ma_val   = X_ma_all[val_normal_ma.values]

X_fc_ma_train = X_ma_train[:, :-1, :]
Y_fc_ma_train = X_ma_train[:, -1, score_idx]
X_fc_ma_val   = X_ma_val[:, :-1, :]
Y_fc_ma_val   = X_ma_val[:, -1, score_idx]

print(f"  MA AE train/val: {X_ma_train.shape} / {X_ma_val.shape}")
print(f"  MA FC train/val: {X_fc_ma_train.shape} / {X_fc_ma_val.shape}")

# --- Heidelberg (Target, injiziert) ---
print("\n--- Heidelberg (Target + Injection) ---")
X_hd_all, meta_hd_all = build_city_sequences(df_hd_inj, scaler_hd, ae_features, cfg, "HD")

meta_hd_all["split_eval"] = np.where(
    meta_hd_all["hour_ts"] < train_end_hd,
    "train",
    np.where(meta_hd_all["hour_ts"] < val_end_hd, "val", "test"),
)

train_normal_hd = (meta_hd_all["split_eval"] == "train") & (meta_hd_all["label_eval"] == "normal")
val_normal_hd   = (meta_hd_all["split_eval"] == "val") & (meta_hd_all["label_eval"] == "normal")

X_hd_train = X_hd_all[train_normal_hd.values]
X_hd_val_clean = X_hd_all[val_normal_hd.values]

X_fc_hd_train = X_hd_train[:, :-1, :]
Y_fc_hd_train = X_hd_train[:, -1, score_idx]
X_fc_hd_val   = X_hd_val_clean[:, :-1, :]
Y_fc_hd_val   = X_hd_val_clean[:, -1, score_idx]

X_fc_hd_all_input = X_hd_all[:, :-1, :]

print(f"  HD AE train/val_clean/all: {X_hd_train.shape} / {X_hd_val_clean.shape} / {X_hd_all.shape}")
print(f"  HD FC train/val_clean/all: {X_fc_hd_train.shape} / {X_fc_hd_val.shape} / {X_fc_hd_all_input.shape}")

# Zero-Shot Variante: HD mit Mannheim-Scaler
print("\n--- Heidelberg Zero-Shot mit Mannheim-Scaler ---")
X_hd_zs_all, meta_hd_zs_all = build_city_sequences(df_hd_inj, scaler_ma, ae_features, cfg, "HD_ZS")
X_hd_zs_fc_input = X_hd_zs_all[:, :-1, :]

# --- Giessen (zusätzliche Source) ---
print("\n--- Gießen (Source extra) ---")
X_gi_all, meta_gi_all = build_city_sequences(df_gi, scaler_gi, ae_features, cfg, "GI")

meta_gi_all["split_eval"] = np.where(
    meta_gi_all["hour_ts"] < train_end_gi,
    "train",
    np.where(meta_gi_all["hour_ts"] < val_end_gi, "val", "test"),
)

train_normal_gi = (meta_gi_all["split_eval"] == "train") & (meta_gi_all["label_eval"] == "normal")
val_normal_gi   = (meta_gi_all["split_eval"] == "val") & (meta_gi_all["label_eval"] == "normal")

X_gi_tr = X_gi_all[train_normal_gi.values]
X_gi_val = X_gi_all[val_normal_gi.values]

X_fc_gi_tr = X_gi_tr[:, :-1, :]
Y_fc_gi_tr = X_gi_tr[:, -1, score_idx]
X_fc_gi_val = X_gi_val[:, :-1, :]
Y_fc_gi_val = X_gi_val[:, -1, score_idx]

# --- Kaiserslautern (zusätzliche Source) ---
print("\n--- Kaiserslautern (Source extra) ---")
X_kl_all, meta_kl_all = build_city_sequences(df_kl, scaler_kl, ae_features, cfg, "KL")

meta_kl_all["split_eval"] = np.where(
    meta_kl_all["hour_ts"] < train_end_kl,
    "train",
    np.where(meta_kl_all["hour_ts"] < val_end_kl, "val", "test"),
)

train_normal_kl = (meta_kl_all["split_eval"] == "train") & (meta_kl_all["label_eval"] == "normal")
val_normal_kl   = (meta_kl_all["split_eval"] == "val") & (meta_kl_all["label_eval"] == "normal")

X_kl_tr = X_kl_all[train_normal_kl.values]
X_kl_val = X_kl_all[val_normal_kl.values]

X_fc_kl_tr = X_kl_tr[:, :-1, :]
Y_fc_kl_tr = X_kl_tr[:, -1, score_idx]
X_fc_kl_val = X_kl_val[:, :-1, :]
Y_fc_kl_val = X_kl_val[:, -1, score_idx]

  ▶ Erzeuge neues injiziertes Target-Dataset...
  ✓ Injiziertes Dataset gespeichert: injected_fd164b228a.parquet
  ✓ Scaler fit: Mannheim | n=1,553,844
  ✓ Scaler fit: Heidelberg | n=1,186,268
  ✓ Scaler fit: Gießen | n=885,524
  ✓ Scaler fit: Kaiserslautern | n=400,992

--- Mannheim (Source) ---
  ✓ MA: Sequenzen gebaut: (2092776, 24, 9)
  MA AE train/val: (1401540, 24, 9) / (314160, 24, 9)
  MA FC train/val: (1401540, 23, 9) / (314160, 23, 9)

--- Heidelberg (Target + Injection) ---
  ✓ HD: Sequenzen gebaut: (1146044, 24, 9)
  HD AE train/val_clean/all: (767510, 24, 9) / (172040, 24, 9) / (1146044, 24, 9)
  HD FC train/val_clean/all: (767510, 23, 9) / (172040, 23, 9) / (1146044, 23, 9)

--- Heidelberg Zero-Shot mit Mannheim-Scaler ---
  ✓ HD_ZS: Sequenzen gebaut: (1146044, 24, 9)

--- Gießen (Source extra) ---
  ✓ GI: Sequenzen gebaut: (847076, 24, 9)

--- Kaiserslautern (Source extra) ---
  ✓ KL: Sequenzen gebaut: (548108, 24, 9)


In [21]:
# ══════════════════════════════════════════════════════════════
# 14 — V17 Source-Modelle laden + Scaler/Config speichern
# ══════════════════════════════════════════════════════════════

print("Lade V17 Source-Modelle (Mannheim)...")

model_ae_ma = keras.models.load_model(V17_AE_PATH)
model_fc_ma = keras.models.load_model(V17_FC_PATH)

print(f"  ✓ AE: {model_ae_ma.count_params():,} Parameter")
print(f"  ✓ FC: {model_fc_ma.count_params():,} Parameter")

# Scaler speichern
for name, scaler in [
    ("mannheim", scaler_ma),
    ("heidelberg", scaler_hd),
    ("gießen", scaler_gi),
    ("kaiserslautern", scaler_kl),
]:
    with open(f"{MODELS_DIR}/scaler_{name}.pkl", "wb") as f:
        pickle.dump(scaler, f)

with open(f"{RESULTS_DIR}/v18a_config.json", "w") as f:
    json.dump(asdict(cfg), f, indent=2, default=str)

splits_info = {
    "MA": {"train_end": str(train_end_ma), "val_end": str(val_end_ma)},
    "HD": {"train_end": str(train_end_hd), "val_end": str(val_end_hd)},
    "GI": {"train_end": str(train_end_gi), "val_end": str(val_end_gi)},
    "KL": {"train_end": str(train_end_kl), "val_end": str(val_end_kl)},
}

with open(f"{RESULTS_DIR}/v18a_splits.json", "w") as f:
    json.dump(splits_info, f, indent=2)

print("  ✓ Config, Splits und Scaler gespeichert.")

Lade V17 Source-Modelle (Mannheim)...
  ✓ AE: 30,633 Parameter
  ✓ FC: 13,762 Parameter
  ✓ Config, Splits und Scaler gespeichert.


---## Kalibrierungs-Experimente (Pipeline-Validierung)Ziel: Technisch pruefen, ob die Pipeline funktioniert. Keine systematische Evaluation.| Check | Modelle | Zweck ||-------|---------|-------|| Oracle (100% Target) | AE + FC | Obere Schranke || Zero-Shot (MA→HD) | AE + FC | Funktioniert Transfer ohne FT? || Fine-Tune 5% (MA→HD) | AE + FC | Single-Source Transfer || Multi-Source FT 5% (MA+GI+KL→HD) | AE + FC | Multi-Source Transfer |**Design-Entscheidungen:**- **Scaler:** Zero-Shot nutzt Source-Scaler, alles andere Target-Scaler- **Freezing:** AE = Encoder frozen, FC = forecast_lstm_1 frozen- **Fine-Tune-Anteil:** 5% als Mittelwert fuer Kalibrierung

In [22]:
# ══════════════════════════════════════════════════════════════
# CAL 1 — Oracle: Target-Only 100% (AE + FC)
# ══════════════════════════════════════════════════════════════

print("=" * 60 + "\n  CAL 1: Oracle (Target-Only 100%)\n" + "=" * 60)

def _build_ae_oracle():
    m = build_lstm_autoencoder(
        len(ae_features),
        cfg.ae_window_size,
        cfg.ae_latent_dim,
        cfg.ae_layers,
        cfg.ae_dropout,
        cfg.use_bidirectional,
        len(ae_features),
    )
    m, h = train_model_generic(
        m,
        X_hd_train,
        X_hd_train,
        X_hd_val_clean,
        X_hd_val_clean,
        cfg.ae_epochs,
        cfg.ae_lr,
        cfg.ae_batch_size,
        cfg.ae_early_stop,
    )
    return m, h

model_ae_oracle, _ = load_or_train(
    V18A_MODEL_FILES["CAL_AE_oracle"],
    _build_ae_oracle,
    FORCE_RETRAIN,
)

X_hat = predict_in_batches(model_ae_oracle, X_hd_all)
meta_hd_all["score_ae_oracle"] = compute_reconstruction_scores(
    X_hd_all, X_hat, ae_features, score_features
)
meta_hd_all, _, _ = znorm_score_by_hour(
    meta_hd_all,
    "score_ae_oracle",
    train_end_hd,
    val_end_hd,
    "score_ae_oracle_z",
)
res = evaluate_v18(
    meta_hd_all,
    "score_ae_oracle_z",
    "CAL_AE_Oracle",
    train_end_hd,
    val_end_hd,
)
results_all = save_result(results_all, "CAL_AE_Oracle", res)

def _build_fc_oracle():
    m = build_lstm_forecaster(
        len(ae_features),
        len(score_features),
        context_steps,
        cfg.ae_latent_dim,
        2,
        cfg.ae_dropout,
    )
    m, h = train_model_generic(
        m,
        X_fc_hd_train,
        Y_fc_hd_train,
        X_fc_hd_val,
        Y_fc_hd_val,
        cfg.ae_epochs,
        cfg.ae_lr,
        cfg.ae_batch_size,
        cfg.ae_early_stop,
    )
    return m, h

model_fc_oracle, _ = load_or_train(
    V18A_MODEL_FILES["CAL_FC_oracle"],
    _build_fc_oracle,
    FORCE_RETRAIN,
)

meta_hd_all["score_fc_oracle"] = compute_forecast_scores(
    model_fc_oracle,
    X_fc_hd_all_input,
    X_hd_all[:, -1, :],
    score_features,
    ae_features,
)
meta_hd_all, _, _ = znorm_score_by_hour(
    meta_hd_all,
    "score_fc_oracle",
    train_end_hd,
    val_end_hd,
    "score_fc_oracle_z",
)
res = evaluate_v18(
    meta_hd_all,
    "score_fc_oracle_z",
    "CAL_FC_Oracle",
    train_end_hd,
    val_end_hd,
)
results_all = save_result(results_all, "CAL_FC_Oracle", res)

print("\n✓ CAL 1 abgeschlossen.")

  CAL 1: Oracle (Target-Only 100%)
  ▶ Trainiere neu: cal_ae_oracle.keras
Epoch 1/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 17s 27ms/step - loss: 0.4314 - val_loss: 0.2997 - learning_rate: 0.0010
Epoch 2/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 8s 22ms/step - loss: 0.2225 - val_loss: 0.1212 - learning_rate: 0.0010
Epoch 3/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 8s 22ms/step - loss: 0.1473 - val_loss: 0.1064 - learning_rate: 0.0010
Epoch 4/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 8s 22ms/step - loss: 0.1330 - val_loss: 0.0995 - learning_rate: 0.0010
Epoch 5/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 8s 22ms/step - loss: 0.1256 - val_loss: 0.0971 - learning_rate: 0.0010
Epoch 6/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 8s 22ms/step - loss: 0.1212 - val_loss: 0.0956 - learning_rate: 0.0010
Epoch 7/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 8s 22ms/step - loss: 0.1177 - val_loss: 0.0943 - learning_rate: 0.0010
Epoch 8/50
375/375 ━━━━━━━━━━━━━━━━━━━━ 8s 22ms/step - loss: 0.1153 - val_loss: 0.0930 - learning_rate: 0.0010
Epoch 9/50
375/375 ━━━━━━━━━━━━━━━━━━

In [23]:
# ══════════════════════════════════════════════════════════════
# CAL 2 — Zero-Shot: MA → HD (kein Training, MA-Scaler)
# ══════════════════════════════════════════════════════════════

print("=" * 60 + "\n  CAL 2: Zero-Shot (MA → HD)\n" + "=" * 60)

# AE Zero-Shot
X_hat_zs = predict_in_batches(model_ae_ma, X_hd_zs_all)
meta_hd_zs_all["score_ae_zs"] = compute_reconstruction_scores(
    X_hd_zs_all, X_hat_zs, ae_features, score_features
)
meta_hd_zs_all, _, _ = znorm_score_by_hour(
    meta_hd_zs_all,
    "score_ae_zs",
    train_end_hd,
    val_end_hd,
    "score_ae_zs_z",
)
res = evaluate_v18(
    meta_hd_zs_all,
    "score_ae_zs_z",
    "CAL_AE_ZeroShot",
    train_end_hd,
    val_end_hd,
)
results_all = save_result(results_all, "CAL_AE_ZeroShot", res)

# FC Zero-Shot
meta_hd_zs_all["score_fc_zs"] = compute_forecast_scores(
    model_fc_ma,
    X_hd_zs_fc_input,
    X_hd_zs_all[:, -1, :],
    score_features,
    ae_features,
)
meta_hd_zs_all, _, _ = znorm_score_by_hour(
    meta_hd_zs_all,
    "score_fc_zs",
    train_end_hd,
    val_end_hd,
    "score_fc_zs_z",
)
res = evaluate_v18(
    meta_hd_zs_all,
    "score_fc_zs_z",
    "CAL_FC_ZeroShot",
    train_end_hd,
    val_end_hd,
)
results_all = save_result(results_all, "CAL_FC_ZeroShot", res)

print("\n✓ CAL 2 abgeschlossen.")

  CAL 2: Zero-Shot (MA → HD)

[CAL_AE_ZeroShot]
  VAL:  n=172,040, positives=0
  TEST: n=206,494, positives=2,307
  Threshold: 3.075950
  PR-AUC:    0.10394893276908122
  F1:        0.2086
  Precision: 0.1519
  Recall:    0.3329

[CAL_FC_ZeroShot]
  VAL:  n=172,040, positives=0
  TEST: n=206,494, positives=2,307
  Threshold: 3.322122
  PR-AUC:    0.2567411820422304
  F1:        0.3251
  Precision: 0.3037
  Recall:    0.3498

✓ CAL 2 abgeschlossen.


In [24]:
# ══════════════════════════════════════════════════════════════
# CAL 3 — Single-Source Fine-Tune 5% (MA → HD)
# ══════════════════════════════════════════════════════════════

print("=" * 60 + "\n  CAL 3: Fine-Tune 5% (MA → HD)\n" + "=" * 60)

frac = 0.05
X_hd_ft, _ = get_last_fraction(X_hd_train, X_hd_train, frac)
X_fc_ft, Y_fc_ft = get_last_fraction_fc(X_fc_hd_train, Y_fc_hd_train, frac)

print(f"Fine-Tune {int(frac * 100)}%: AE={X_hd_ft.shape}, FC={X_fc_ft.shape}")

# AE
def _build_ft_ae_5pct():
    m = keras.models.load_model(V17_AE_PATH)
    m = freeze_ae_encoder(m)
    m, h = fine_tune_model(
        m,
        X_hd_ft,
        X_hd_ft,
        X_hd_val_clean,
        X_hd_val_clean,
        cfg.ft_epochs,
        cfg.ft_lr,
        cfg.ft_batch_size,
        cfg.ft_early_stop,
    )
    return m, h

model_ae_ft, _ = load_or_train(
    V18A_MODEL_FILES["CAL_AE_FT_5pct"],
    _build_ft_ae_5pct,
    FORCE_RETRAIN,
)

X_hat = predict_in_batches(model_ae_ft, X_hd_all)
meta_hd_all["score_ae_ft5"] = compute_reconstruction_scores(
    X_hd_all, X_hat, ae_features, score_features
)
meta_hd_all, _, _ = znorm_score_by_hour(
    meta_hd_all,
    "score_ae_ft5",
    train_end_hd,
    val_end_hd,
    "score_ae_ft5_z",
)
res = evaluate_v18(
    meta_hd_all,
    "score_ae_ft5_z",
    "CAL_AE_FT_5pct",
    train_end_hd,
    val_end_hd,
)
results_all = save_result(results_all, "CAL_AE_FT_5pct", res)

# FC
def _build_ft_fc_5pct():
    m = keras.models.load_model(V17_FC_PATH)
    m = freeze_fc_backbone(m)
    m, h = fine_tune_model(
        m,
        X_fc_ft,
        Y_fc_ft,
        X_fc_hd_val,
        Y_fc_hd_val,
        cfg.ft_epochs,
        cfg.ft_lr,
        cfg.ft_batch_size,
        cfg.ft_early_stop,
    )
    return m, h

model_fc_ft, _ = load_or_train(
    V18A_MODEL_FILES["CAL_FC_FT_5pct"],
    _build_ft_fc_5pct,
    FORCE_RETRAIN,
)

meta_hd_all["score_fc_ft5"] = compute_forecast_scores(
    model_fc_ft,
    X_fc_hd_all_input,
    X_hd_all[:, -1, :],
    score_features,
    ae_features,
)
meta_hd_all, _, _ = znorm_score_by_hour(
    meta_hd_all,
    "score_fc_ft5",
    train_end_hd,
    val_end_hd,
    "score_fc_ft5_z",
)
res = evaluate_v18(
    meta_hd_all,
    "score_fc_ft5_z",
    "CAL_FC_FT_5pct",
    train_end_hd,
    val_end_hd,
)
results_all = save_result(results_all, "CAL_FC_FT_5pct", res)

print("\n✓ CAL 3 abgeschlossen.")

  CAL 3: Fine-Tune 5% (MA → HD)
Fine-Tune 5%: AE=(38375, 24, 9), FC=(38375, 23, 9)
  ▶ Trainiere neu: cal_ae_ft_5pct.keras
  AE Freeze: 13,696 frozen, 16,937 trainable
Epoch 1/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 8s 53ms/step - loss: 0.0967 - val_loss: 0.0746
Epoch 2/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - loss: 0.0953 - val_loss: 0.0753
Epoch 3/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - loss: 0.0951 - val_loss: 0.0759
Epoch 4/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - loss: 0.0945 - val_loss: 0.0761
Epoch 5/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - loss: 0.0942 - val_loss: 0.0765
Epoch 6/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - loss: 0.0938 - val_loss: 0.0771
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 1.
  ✓ Gespeichert: cal_ae_ft_5pct.keras

[CAL_AE_FT_5pct]
  VAL:  n=172,040, positives=0
  TEST: n=206,494, positives=2,307
  Threshold: 3.185269
  PR-AUC:    0.10113697743232938
  F1:        0.2079
  Precision: 0.1526
  Recall:    0.3264
  

In [25]:
# ══════════════════════════════════════════════════════════════
# CAL 4 — Multi-Source Fine-Tune 5% (MA + GI + KL → HD)
# ══════════════════════════════════════════════════════════════

print("=" * 60 + "\n  CAL 4: Multi-Source Fine-Tune 5% (MA+GI+KL → HD)\n" + "=" * 60)

# --- FC Multi-Source ---
X_fc_multi_tr = np.concatenate([X_fc_ma_train, X_fc_gi_tr, X_fc_kl_tr], axis=0)
Y_fc_multi_tr = np.concatenate([Y_fc_ma_train, Y_fc_gi_tr, Y_fc_kl_tr], axis=0)
X_fc_multi_val = np.concatenate([X_fc_ma_val, X_fc_gi_val, X_fc_kl_val], axis=0)
Y_fc_multi_val = np.concatenate([Y_fc_ma_val, Y_fc_gi_val, Y_fc_kl_val], axis=0)

print(f"Multi-Source FC — Train: {X_fc_multi_tr.shape}, Val: {X_fc_multi_val.shape}")

def _build_fc_multi():
    m = build_lstm_forecaster(
        len(ae_features),
        len(score_features),
        context_steps,
        cfg.ae_latent_dim,
        2,
        cfg.ae_dropout,
    )
    m, h = train_model_generic(
        m,
        X_fc_multi_tr,
        Y_fc_multi_tr,
        X_fc_multi_val,
        Y_fc_multi_val,
        cfg.ae_epochs,
        cfg.ae_lr,
        cfg.ae_batch_size,
        cfg.ae_early_stop,
    )
    return m, h

model_fc_multi, _ = load_or_train(
    V18A_MODEL_FILES["CAL_FC_multi_source"],
    _build_fc_multi,
    FORCE_RETRAIN,
)

frac = 0.05
X_fc_m_ft, Y_fc_m_ft = get_last_fraction_fc(X_fc_hd_train, Y_fc_hd_train, frac)

def _build_ft_fc_multi_5pct():
    m = keras.models.load_model(V18A_MODEL_FILES["CAL_FC_multi_source"])
    m = freeze_fc_backbone(m)
    m, h = fine_tune_model(
        m,
        X_fc_m_ft,
        Y_fc_m_ft,
        X_fc_hd_val,
        Y_fc_hd_val,
        cfg.ft_epochs,
        cfg.ft_lr,
        cfg.ft_batch_size,
        cfg.ft_early_stop,
    )
    return m, h

model_fc_multi_ft, _ = load_or_train(
    V18A_MODEL_FILES["CAL_FC_Multi_FT_5pct"],
    _build_ft_fc_multi_5pct,
    FORCE_RETRAIN,
)

meta_hd_all["score_fc_multi_ft5"] = compute_forecast_scores(
    model_fc_multi_ft,
    X_fc_hd_all_input,
    X_hd_all[:, -1, :],
    score_features,
    ae_features,
)
meta_hd_all, _, _ = znorm_score_by_hour(
    meta_hd_all,
    "score_fc_multi_ft5",
    train_end_hd,
    val_end_hd,
    "score_fc_multi_ft5_z",
)
res = evaluate_v18(
    meta_hd_all,
    "score_fc_multi_ft5_z",
    "CAL_FC_Multi_FT_5pct",
    train_end_hd,
    val_end_hd,
)
results_all = save_result(results_all, "CAL_FC_Multi_FT_5pct", res)

# --- AE Multi-Source ---
X_ae_multi_tr = np.concatenate([X_ma_train, X_gi_tr, X_kl_tr], axis=0)
X_ae_multi_val = np.concatenate([X_ma_val, X_gi_val, X_kl_val], axis=0)

print(f"Multi-Source AE — Train: {X_ae_multi_tr.shape}, Val: {X_ae_multi_val.shape}")

def _build_ae_multi():
    m = build_lstm_autoencoder(
        len(ae_features),
        cfg.ae_window_size,
        cfg.ae_latent_dim,
        cfg.ae_layers,
        cfg.ae_dropout,
        cfg.use_bidirectional,
        len(ae_features),
    )
    m, h = train_model_generic(
        m,
        X_ae_multi_tr,
        X_ae_multi_tr,
        X_ae_multi_val,
        X_ae_multi_val,
        cfg.ae_epochs,
        cfg.ae_lr,
        cfg.ae_batch_size,
        cfg.ae_early_stop,
    )
    return m, h

model_ae_multi, _ = load_or_train(
    V18A_MODEL_FILES["CAL_AE_multi_source"],
    _build_ae_multi,
    FORCE_RETRAIN,
)

X_hd_m_ft, _ = get_last_fraction(X_hd_train, X_hd_train, frac)

def _build_ft_ae_multi_5pct():
    m = keras.models.load_model(V18A_MODEL_FILES["CAL_AE_multi_source"])
    m = freeze_ae_encoder(m)
    m, h = fine_tune_model(
        m,
        X_hd_m_ft,
        X_hd_m_ft,
        X_hd_val_clean,
        X_hd_val_clean,
        cfg.ft_epochs,
        cfg.ft_lr,
        cfg.ft_batch_size,
        cfg.ft_early_stop,
    )
    return m, h

model_ae_multi_ft, _ = load_or_train(
    V18A_MODEL_FILES["CAL_AE_Multi_FT_5pct"],
    _build_ft_ae_multi_5pct,
    FORCE_RETRAIN,
)

X_hat = predict_in_batches(model_ae_multi_ft, X_hd_all)
meta_hd_all["score_ae_multi_ft5"] = compute_reconstruction_scores(
    X_hd_all, X_hat, ae_features, score_features
)
meta_hd_all, _, _ = znorm_score_by_hour(
    meta_hd_all,
    "score_ae_multi_ft5",
    train_end_hd,
    val_end_hd,
    "score_ae_multi_ft5_z",
)
res = evaluate_v18(
    meta_hd_all,
    "score_ae_multi_ft5_z",
    "CAL_AE_Multi_FT_5pct",
    train_end_hd,
    val_end_hd,
)
results_all = save_result(results_all, "CAL_AE_Multi_FT_5pct", res)

print("\n✓ CAL 4 abgeschlossen.")

  CAL 4: Multi-Source Fine-Tune 5% (MA+GI+KL → HD)
Multi-Source FC — Train: (2335900, 23, 9), Val: (523600, 23, 9)
  ▶ Trainiere neu: cal_fc_multi_source.keras
Epoch 1/50
1141/1141 ━━━━━━━━━━━━━━━━━━━━ 15s 12ms/step - loss: 0.5301 - val_loss: 0.5975 - learning_rate: 0.0010
Epoch 2/50
1141/1141 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - loss: 0.4967 - val_loss: 0.5875 - learning_rate: 0.0010
Epoch 3/50
1141/1141 ━━━━━━━━━━━━━━━━━━━━ 12s 11ms/step - loss: 0.4865 - val_loss: 0.5799 - learning_rate: 0.0010
Epoch 4/50
1141/1141 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - loss: 0.4804 - val_loss: 0.5746 - learning_rate: 0.0010
Epoch 5/50
1141/1141 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - loss: 0.4765 - val_loss: 0.5705 - learning_rate: 0.0010
Epoch 6/50
1141/1141 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - loss: 0.4732 - val_loss: 0.5700 - learning_rate: 0.0010
Epoch 7/50
1141/1141 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - loss: 0.4690 - val_loss: 0.5616 - learning_rate: 0.0010
Epoch 8/50
1141/1141 ━━━━━━━━━━━━━━━━━━━━ 

In [26]:
# ══════════════════════════════════════════════════════════════
# Ergebniszusammenfassung — Konsolenprint
# ══════════════════════════════════════════════════════════════

def _fmt(x, nd=4):
    if x is None:
        return "N/A"
    try:
        if pd.isna(x):
            return "N/A"
    except Exception:
        pass
    try:
        return f"{float(x):.{nd}f}"
    except Exception:
        return str(x)

def _safe_get(d, *keys, default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

def print_v18a_summary(results_all):
    print("\n" + "=" * 110)
    print(" V18a — ERGEBNISZUSAMMENFASSUNG")
    print("=" * 110)

    order = [
        ("CAL_AE_Oracle",        "AE", "Oracle",    "HD",       "100%"),
        ("CAL_AE_ZeroShot",      "AE", "Zero-Shot", "MA",       "0%"),
        ("CAL_AE_FT_5pct",       "AE", "Fine-Tune", "MA",       "5%"),
        ("CAL_AE_Multi_FT_5pct", "AE", "Multi-FT",  "MA+GI+KL", "5%"),
        ("CAL_FC_Oracle",        "FC", "Oracle",    "HD",       "100%"),
        ("CAL_FC_ZeroShot",      "FC", "Zero-Shot", "MA",       "0%"),
        ("CAL_FC_FT_5pct",       "FC", "Fine-Tune", "MA",       "5%"),
        ("CAL_FC_Multi_FT_5pct", "FC", "Multi-FT",  "MA+GI+KL", "5%"),
    ]

    print(
        f"{'Experiment':<30} {'M':<3} {'Modus':<10} {'Source':<10} {'Tgt%':<5} "
        f"{'PR-AUC':>8} {'F1':>8} {'Prec':>8} {'Recall':>8} {'Thr':>10}"
    )
    print("-" * 110)

    rows = []
    for key, mdl, mode, src, tgt in order:
        r = results_all.get(key, {})
        row = {
            "key": key,
            "model": mdl,
            "mode": mode,
            "source": src,
            "target_pct": tgt,
            "pr_auc": r.get("pr_auc"),
            "f1": r.get("f1"),
            "precision": r.get("precision"),
            "recall": r.get("recall"),
            "threshold": r.get("threshold"),
        }
        rows.append(row)

        print(
            f"{key:<30} {mdl:<3} {mode:<10} {src:<10} {tgt:<5} "
            f"{_fmt(row['pr_auc']):>8} {_fmt(row['f1']):>8} {_fmt(row['precision']):>8} "
            f"{_fmt(row['recall']):>8} {_fmt(row['threshold'], 6):>10}"
        )

    # Bestes Modell gesamt nach PR-AUC
    valid_pr = [r for r in rows if r["pr_auc"] is not None and not pd.isna(r["pr_auc"])]
    if valid_pr:
        best = max(valid_pr, key=lambda x: x["pr_auc"])
        print("\n" + "-" * 110)
        print(" Bestes Experiment nach PR-AUC")
        print("-" * 110)
        print(
            f" {best['key']} | Modell={best['model']} | Modus={best['mode']} | "
            f"PR-AUC={_fmt(best['pr_auc'])} | F1={_fmt(best['f1'])} | Recall={_fmt(best['recall'])}"
        )

    # Bestes AE / FC
    for fam in ["AE", "FC"]:
        fam_rows = [r for r in rows if r["model"] == fam and r["pr_auc"] is not None and not pd.isna(r["pr_auc"])]
        if fam_rows:
            best_fam = max(fam_rows, key=lambda x: x["pr_auc"])
            print(
                f" Bestes {fam}: {best_fam['key']} "
                f"(PR-AUC={_fmt(best_fam['pr_auc'])}, F1={_fmt(best_fam['f1'])}, Recall={_fmt(best_fam['recall'])})"
            )

    # Detailansicht pro Experiment
    print("\n" + "=" * 110)
    print(" DETAILANSICHT: PER-TYPE UND PER-REGIME RECALL")
    print("=" * 110)

    for key, mdl, mode, src, tgt in order:
        r = results_all.get(key, {})
        if not r:
            continue

        print(f"\n[{key}]  Modell={mdl} | Modus={mode} | Source={src} | Target={tgt}")
        print(
            f"  PR-AUC={_fmt(r.get('pr_auc'))} | "
            f"F1={_fmt(r.get('f1'))} | "
            f"Precision={_fmt(r.get('precision'))} | "
            f"Recall={_fmt(r.get('recall'))} | "
            f"Threshold={_fmt(r.get('threshold'), 6)}"
        )

        per_type = r.get("per_type", {})
        if per_type:
            print("  Per-Type Recall:")
            for t in ["point_spike", "point_drop", "collective"]:
                rr = per_type.get(t)
                if rr:
                    print(f"    - {t:<12} n={rr.get('n', 0):>6} | recall={_fmt(rr.get('recall'))}")
                else:
                    print(f"    - {t:<12} n={'0':>6} | recall=N/A")

        per_regime = r.get("per_regime", {})
        if per_regime:
            print("  Per-Regime Recall:")
            for reg in ["high", "mid", "low"]:
                rr = per_regime.get(reg)
                if rr:
                    print(f"    - {reg:<12} n={rr.get('n', 0):>6} | recall={_fmt(rr.get('recall'))}")
                else:
                    print(f"    - {reg:<12} n={'0':>6} | recall=N/A")

    print("\n" + "=" * 110)
    print(" ENDE DER ZUSAMMENFASSUNG")
    print("=" * 110)

# Aufruf
print_v18a_summary(results_all)


 V18a — ERGEBNISZUSAMMENFASSUNG
Experiment                     M   Modus      Source     Tgt%    PR-AUC       F1     Prec   Recall        Thr
--------------------------------------------------------------------------------------------------------------
CAL_AE_Oracle                  AE  Oracle     HD         100%    0.1023   0.2133   0.1615   0.3143   3.790710
CAL_AE_ZeroShot                AE  Zero-Shot  MA         0%      0.1039   0.2086   0.1519   0.3329   3.075950
CAL_AE_FT_5pct                 AE  Fine-Tune  MA         5%      0.1011   0.2079   0.1526   0.3264   3.185269
CAL_AE_Multi_FT_5pct           AE  Multi-FT   MA+GI+KL   5%      0.1075   0.2110   0.1552   0.3294   3.340081
CAL_FC_Oracle                  FC  Oracle     HD         100%    0.2670   0.3344   0.3104   0.3624   3.368177
CAL_FC_ZeroShot                FC  Zero-Shot  MA         0%      0.2567   0.3251   0.3037   0.3498   3.322122
CAL_FC_FT_5pct                 FC  Fine-Tune  MA         5%      0.2528   0.3237   0.3

## Kalibrierungs-Zusammenfassung + Config-Freeze

In [16]:
# ══════════════════════════════════════════════════════════════
# Kalibrierungs-Ergebnistabelle
# ══════════════════════════════════════════════════════════════

results_all = load_results_cache()

CAL_ORDER = [
    ("CAL_AE_Oracle",        "AE", "Oracle",    "HD",        "100%"),
    ("CAL_AE_ZeroShot",      "AE", "Zero-Shot", "MA",        "0%"),
    ("CAL_AE_FT_5pct",       "AE", "Fine-Tune", "MA",        "5%"),
    ("CAL_AE_Multi_FT_5pct", "AE", "Multi-FT",  "MA+GI+KL",  "5%"),
    ("CAL_FC_Oracle",        "FC", "Oracle",    "HD",        "100%"),
    ("CAL_FC_ZeroShot",      "FC", "Zero-Shot", "MA",        "0%"),
    ("CAL_FC_FT_5pct",       "FC", "Fine-Tune", "MA",        "5%"),
    ("CAL_FC_Multi_FT_5pct", "FC", "Multi-FT",  "MA+GI+KL",  "5%"),
]

def fmt(v):
    return f"{v:.4f}" if v is not None and not (isinstance(v, float) and np.isnan(v)) else "N/A"

print("=" * 100)
print("  V18a KALIBRIERUNG — ERGEBNISUEBERSICHT")
print("=" * 100)
print(
    f"{'Experiment':<30} {'Mod':3} {'Modus':<12} {'Source':<10} "
    f"{'Tgt%':5} {'PR-AUC':>8} {'F1':>8} {'Prec':>8} {'Recall':>8}"
)
print("-" * 100)

for key, mdl, mode, src, tgt_pct in CAL_ORDER:
    r = results_all.get(key, {})
    print(
        f"{key:<30} {mdl:3} {mode:<12} {src:<10} {tgt_pct:5} "
        f"{fmt(r.get('pr_auc')):>8} {fmt(r.get('f1')):>8} "
        f"{fmt(r.get('precision')):>8} {fmt(r.get('recall')):>8}"
    )

  V18a KALIBRIERUNG — ERGEBNISUEBERSICHT
Experiment                     Mod Modus        Source     Tgt%    PR-AUC       F1     Prec   Recall
----------------------------------------------------------------------------------------------------
CAL_AE_Oracle                  AE  Oracle       HD         100%    0.1023   0.2133   0.1615   0.3143
CAL_AE_ZeroShot                AE  Zero-Shot    MA         0%      0.1039   0.2086   0.1519   0.3329
CAL_AE_FT_5pct                 AE  Fine-Tune    MA         5%      0.1011   0.2079   0.1526   0.3264
CAL_AE_Multi_FT_5pct           AE  Multi-FT     MA+GI+KL   5%      0.1075   0.2110   0.1552   0.3294
CAL_FC_Oracle                  FC  Oracle       HD         100%    0.2670   0.3344   0.3104   0.3624
CAL_FC_ZeroShot                FC  Zero-Shot    MA         0%      0.2567   0.3251   0.3037   0.3498
CAL_FC_FT_5pct                 FC  Fine-Tune    MA         5%      0.2528   0.3237   0.3035   0.3468
CAL_FC_Multi_FT_5pct           FC  Multi-FT     MA

In [ ]:
# ══════════════════════════════════════════════════════════════# Alle Artefakte speichern — Pipeline fixiert# ══════════════════════════════════════════════════════════════with open(f"{RESULTS_DIR}/v18a_all_results.json", "w") as f:    json.dump(results_all, f, indent=2, default=str)# Config als fixiert markierencfg_frozen = asdict(cfg)cfg_frozen["_frozen"] = Truecfg_frozen["_calibration_complete"] = Truewith open(f"{RESULTS_DIR}/v18a_config_frozen.json", "w") as f:    json.dump(cfg_frozen, f, indent=2, default=str)print("="*60)print("  V18a ARTEFAKTE GESPEICHERT — PIPELINE FIXIERT")print("="*60)print(f"  Config: {RESULTS_DIR}/v18a_config_frozen.json")print(f"  Results: {RESULTS_DIR}/v18a_all_results.json")for f_path in sorted(glob.glob(f"{MODELS_DIR}/*")):    sz = os.path.getsize(f_path) / 1e6    print(f"  ✓ {os.path.basename(f_path):45s} {sz:6.1f} MB")print("\n  → Naechster Schritt: V18b (Laborexperiment) mit fixierter Config starten.")